In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline

In [ ]:

# 1. Load text from file (same as in Pipeline A)
def load_text(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

In [ ]:

# 2. Segment text into chunks (e.g., every 1000 words)
def segment_text(text, words_per_segment=1000):
    # Use regex to extract words (all lowercase)
    words = re.findall(r'\w+', text.lower())
    segments = []
    for i in range(0, len(words), words_per_segment):
        segment = " ".join(words[i:i+words_per_segment])
        segments.append(segment)
    return segments

In [ ]:

# 3. Initialize a sentiment analysis pipeline from Hugging Face.
#    (We use a model fine-tuned for sentiment classification.)
def init_sentiment_pipeline():
    sentiment_pipeline = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
    return sentiment_pipeline

In [ ]:

# 4. Process each segment to obtain sentiment scores.
def analyze_sentiments(segments, sentiment_pipeline):
    # For each segment, obtain a sentiment analysis result.
    # The model outputs a list with a dict: e.g. {"label": "POSITIVE", "score": 0.99}
    sentiment_scores = []
    for segment in segments:
        result = sentiment_pipeline(segment[:512])  # Limit input size if necessary
        # We'll convert the result to a numeric score: positive => +score, negative => -score.
        if result[0]['label'] == 'POSITIVE':
            score = result[0]['score']
        else:
            score = -result[0]['score']
        sentiment_scores.append(score)
    return np.array(sentiment_scores)

In [ ]:

# 5. Optionally, smooth the sentiment signal to capture the overall trend.
def smooth_signal(signal, window_size=3):
    if window_size < 2:
        return signal
    smoothed = np.convolve(signal, np.ones(window_size)/window_size, mode='valid')
    # To match original length, we can pad at the beginning.
    pad_width = len(signal) - len(smoothed)
    smoothed = np.pad(smoothed, (pad_width, 0), mode='edge')
    return smoothed

In [ ]:

# 6. Visualization: plot the sentiment scores across segments.
def plot_sentiment(segments_range, sentiment_scores, smoothed_scores):
    plt.figure(figsize=(10, 6))
    plt.plot(segments_range, sentiment_scores, label='Segment Sentiment Score', marker='o', linestyle='--', alpha=0.7)
    plt.plot(segments_range, smoothed_scores, label='Smoothed Sentiment Trend', color='red', linewidth=2)
    plt.xlabel('Segment Index')
    plt.ylabel('Sentiment Score')
    plt.title('LLM-Based Sentiment Analysis of The Foucault Pendulum')
    plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

# 7. Main function for Pipeline B
def pipeline_b(filename, words_per_segment=1000):
    # Load and segment the text
    text = load_text(filename)
    segments = segment_text(text, words_per_segment)
    segments_range = np.arange(len(segments))

    # Initialize the sentiment analysis pipeline
    sentiment_pipeline = init_sentiment_pipeline()

    # Analyze each segment to get sentiment scores
    sentiment_scores = analyze_sentiments(segments, sentiment_pipeline)

    # Smooth the sentiment signal (optional)
    smoothed_scores = smooth_signal(sentiment_scores, window_size=3)

    # Visualization of the sentiment scores over segments
    plot_sentiment(segments_range, sentiment_scores, smoothed_scores)

    # Return computed sentiment scores for further analysis
    return {
        'sentiment_scores': sentiment_scores,
        'smoothed_scores': smoothed_scores,
        'segments': segments
    }

In [ ]:

# 8. Execute the pipeline if run as a script
if __name__ == "__main__":
    # Replace 'the_foucault_pendulum.txt' with the path to your text file
    results = pipeline_b('the_foucault_pendulum.txt', words_per_segment=1000)
